# LegalGPT: train and benchmark the 1.5B on exactly the 3B/7B training data (Colab L4, ~40 min)

So the three model sizes are compared fairly, the 1.5B learns from **the same examples and split**
as the 3B: `MyDrive/legalgpt/lora/v0.7-3b-l4/examples.jsonl`, written by the earlier run.

1. Upload the **new** `legalgpt_colab.zip` to `MyDrive/legalgpt/`, replacing the old one.
2. **Runtime → Change runtime type → L4 GPU**, then **Run all** (about 40 min, roughly 3-4 units).
3. Afterwards, download `MyDrive/legalgpt/bench/legalgpt-lora_v0.7-1.5b.jsonl` into the laptop's
   `data/reports/bench/baselines/`, and the folder `MyDrive/legalgpt/lora/v0.7-1.5b-l4` (without
   `_trainer`) into `data/models/lora/`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, subprocess, shutil, zipfile
DRIVE, REPO = "/content/drive/MyDrive/legalgpt", "/content/legaltech"
EXAMPLES = f"{DRIVE}/lora/v0.7-3b-l4/examples.jsonl"
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print(gpu)
assert "T4" not in gpu, "Change the runtime to L4 (Runtime -> Change runtime type)"
assert os.path.exists(EXAMPLES), f"{EXAMPLES} is missing"
shutil.rmtree(REPO, ignore_errors=True)
zipfile.ZipFile(f"{DRIVE}/legalgpt_colab.zip").extractall(REPO)
shutil.copy(EXAMPLES, "/content/examples.jsonl")   # read from Colab's own disk, not Drive
!pip install -q -r {REPO}/backend/scripts/colab/requirements-colab.txt

In [ ]:
def sh(cmd):
    """Run a command, stream its output, and stop the notebook if it fails."""
    env = {**os.environ, "TQDM_MININTERVAL": "60", "PYTHONUNBUFFERED": "1"}
    p = subprocess.Popen(cmd, shell=True, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"command failed ({p.returncode})")

out = f"{DRIVE}/lora/v0.7-1.5b-l4"
sh("python backend/scripts/train_lora.py --base-model Qwen/Qwen2.5-1.5B-Instruct --version v0.2 --no-judgments "
   "--examples-from /content/examples.jsonl --epochs 2 --batch-size 4 --grad-accum 2 --save-steps 20 "
   f"--out {out}")
print(sorted(os.listdir(out)))

## Benchmark (150 questions, fully offline: the model rewrites each question itself)
4-bit, as the 3B was benchmarked. The adapter is copied to Colab's own disk first, and **4 test
questions run before the full run**: any failure stops the notebook here with the error shown.

In [ ]:
# the adapter on Colab's own disk (reading it from Drive is the suspected cause of the earlier 3B failure)
shutil.copytree(out, "/content/adapter-1.5b", ignore=shutil.ignore_patterns("_trainer"), dirs_exist_ok=True)
os.makedirs(f"{DRIVE}/bench", exist_ok=True)
os.makedirs(f"{REPO}/data/reports/bench", exist_ok=True)
if not os.path.exists(f"{REPO}/data/reports/bench/baselines"):
    os.symlink(f"{DRIVE}/bench", f"{REPO}/data/reports/bench/baselines")   # answers saved to Drive as they come
os.environ.update({"ADAPTER_PATH": "/content/adapter-1.5b", "LOCAL_LLM_DTYPE": "4bit"})

def run(cmd):
    """Like sh(), but returns the output so the test can be checked."""
    env = {**os.environ, "TQDM_MININTERVAL": "60", "PYTHONUNBUFFERED": "1"}
    p = subprocess.Popen(cmd, shell=True, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in p.stdout:
        print(line, end=""); lines.append(line)
    if p.wait() != 0:
        raise RuntimeError(f"command failed ({p.returncode})")
    return "".join(lines)

smoke = run("python backend/scripts/run_baselines.py run --smoke --rewrite --suffix _v0.7-1.5b --systems legalgpt-lora")
assert "[FAILED]" not in smoke and smoke.count("[ok]") == 4, "the test questions failed: the error is in the answers above"
print("test questions OK - starting the full run (progress every 10 answers)")
run("python backend/scripts/run_baselines.py run --rewrite --suffix _v0.7-1.5b --systems legalgpt-lora")
run("python backend/scripts/run_baselines.py score | grep -E 'system|lora'")

In [ ]:
# stop the GPU so no more compute units are used (the adapter is in Drive)
from google.colab import runtime
runtime.unassign()